# RetailPulse — exploration

Run `retailpulse download && retailpulse run` first. This notebook reads the scored warehouse and walks through the main results.

In [ ]:
import json
from pathlib import Path

import duckdb
import plotly.express as px

from retailpulse.config import Settings

s = Settings()
con = duckdb.connect(str(s.warehouse_path), read_only=True)
m = json.loads(s.metrics_path.read_text())
m['data']

## Revenue trend

In [ ]:
monthly = con.sql('SELECT * FROM analytics.monthly_kpis ORDER BY month').df()
px.line(monthly, x='month', y=['revenue'], title='Monthly revenue')

## RFM segments

In [ ]:
seg = con.sql('''
    SELECT customer_segment, COUNT(*) customers, SUM(total_revenue) revenue,
           AVG(predicted_12m_clv) avg_clv, AVG(churn_probability) avg_churn
    FROM analytics.customer_360_clv GROUP BY 1 ORDER BY revenue DESC''').df()
seg

In [ ]:
px.bar(seg, x='customer_segment', y='revenue', title='Revenue by segment')

## Churn model validation (expanding window)

In [ ]:
import pandas as pd
pd.DataFrame(m['churn']['cv_summary']).set_index('model')[['roc_auc_mean', 'roc_auc_std', 'pr_auc_return_mean', 'return_lift_top10pct_mean']]

In [ ]:
imp = pd.DataFrame(m['churn']['feature_importance'])
px.bar(imp[::-1], x='importance_mean', y='feature', orientation='h', title='Permutation importance (Δ ROC-AUC)')

## CLV backtest
BG/NBD + Gamma-Gamma vs the original heuristic, scored on a held-out window.

In [ ]:
bt = m['clv']['backtest']
pd.DataFrame({'BG/NBD + Gamma-Gamma': bt['bgnbd_gamma_gamma'], 'Heuristic proxy': bt['heuristic_proxy']})

## Retention priority

In [ ]:
pd.DataFrame(m['priority']).set_index('retention_priority')

In [ ]:
pts = con.sql('SELECT predicted_12m_clv, churn_probability, retention_priority FROM analytics.customer_360_clv USING SAMPLE 5000 ROWS').df()
px.scatter(pts, x='churn_probability', y='predicted_12m_clv', color='retention_priority', log_y=True, opacity=0.6)